In [ ]:
# from pyfastpfor import *
# # Get the list of all codecs
# # getCodecList()

In [ ]:
import sys
sys.path.append('/home/fish2/mdf_compression')
from utils import zigzag_encode, zigzag_decode
from utils.compress import compress_array
from utils.decompress import decompress_array
from utils.serialize import mdfc_writer, mdfc_reader
from utils import create_unified_timeaxis, transform_timestamps, map_times_to_timeaxis

In [ ]:
import asammdf, numpy as np
from io import BytesIO
# sample_data_path = '../sample_data/sample_data.mf4'
sample_data_path = '../sample_data/sample_data_high_random_values.mf4'

testfil = '/home/fish2/mdf_compression/examples/testing.mdfc'

In [ ]:
# channel_name = ['1000ms_9']  # single example
vals = {}  # name: vals
tmss = {}  # name: tmss
with (
    asammdf.MDF(sample_data_path) as mfil, 
    mdfc_writer(testfil, overwrite=True) as writer
):
    # TODO timeaxis could go at the end, to avoid reading all of it, 
    #   but, just a further optimization
    all_times = create_unified_timeaxis(mfil)  # float64 raw times, us -> save to map index of
    # write all times
    writer.append_time_axis(all_times)  # compresses, writes, saves flag

    # now, for each signal, iteratively add it to the compressor
    # print(mfil.channels_db)
    channels = mfil.channels_db
    channels.pop('time')  # done upfront
    
    for sgname, chan in channels.items():
        # print(sgname, chan)
        # TODO need to enhance metadata for multiple names in channels
        #   we take name->channel approach, this is "wrong", 
        #   need to take group->channel->name or whatever approach
        sg = mfil.select([(sgname, *chan[0])], raw=True)[0]
        # time
        timestamps = sg.timestamps  # this is done in map_times_to_timeaxis now
        # get the index positions of the time
        timelocs = map_times_to_timeaxis(timestamps, all_times).astype(np.uint32)  # index position
        # single differentiate 
        timelocs = np.diff(timelocs, prepend=0).astype(np.uint32)

        # value
        samples = sg.samples
        # zigzag & vstack timelocs
        # TODO cleanup & put elsewhere
        #   and then enhance for other dtypes
        samples = zigzag_encode(samples).astype(np.uint32)
        # stack & flatten
        stacked = np.vstack((timelocs, samples)).flatten().astype(np.uint32)
        stacked = samples

        # write the values_block
        #   which contains the differentiated index position of the timeblock
        writer.append_values_block(stacked, sgname)
    # finalize the writer by adding metadata
    writer.finish()


In [ ]:
all_times

In [ ]:
raw_times = np.diff(all_times, prepend=0)#.astype(np.uint32)
raw_times

In [ ]:
raw_times.min()

In [ ]:
cmp_times = compress_array(raw_times)

In [ ]:
all_times[-8:]

In [ ]:
3599900000000 - 3599800000000

In [ ]:
np.diff(np.diff(all_times[-8:], prepend=0).astype(np.uint64), prepend=0).astype(np.uint64)

In [ ]:
raw_times

In [ ]:
cmp_times

In [ ]:
decomp_times = decompress_array(cmp_times, 43200)
decomp_times

In [ ]:
with mdfc_reader(testfil) as reader:
    md = reader.metadata
    reader.load_timeaxis()
    timeaxis = reader.timeaxis

In [ ]:
timeaxis

In [ ]:
md

In [ ]:
# ziggyzaggy this, and ensure we can convert it to uint32, 
#   which will cause an issue if it does not fit in 32bits!
#   TODO something for later :)
vals = zigzag_encode(vals).astype(np.uint32)

In [ ]:
vals

In [ ]:
# double-differentiate it and make uint32
timepos = np.diff(np.diff(timepos, prepend=0), prepend=0).astype(np.uint32)

In [ ]:
timepos

In [ ]:
# now we need to smush these axes together
# this is just np stack method, 
# but, TODO we will need to think about how this stacks with 2d arrays
#   it is OK for now i guess? we can save the resulting shape of it
#   and maybe also save the timeaxis position of it 
combined = np.vstack(
        (
            timepos, 
            vals
        )
    )

In [ ]:
combined

In [ ]:
# now we can compress this?
compd = compress_array(combined.flatten())
compd

In [ ]:
decompd = decompress_array(compd, 3600*2)

In [ ]:
np.reshape(decompd, (2, 3600))

In [ ]:
compd.__sizeof__(), decompd.__sizeof__()

In [ ]:
9748/28912

In [ ]:
all_times

In [ ]:
t1 = np.diff(timepos, prepend=0).astype(np.uint32)

In [ ]:
np.cumsum(t1).astype(np.uint64)

In [ ]:
%%timeit
all_times[timepos]

In [ ]:
# we can apply some transformations of time, 
#   1) convert into uint and seconds -> ns
#   2) take two differentials to ensure it fits within uint32
#       and is small values
#   3) compress using fastpfor 

for key, tm in tmss.items():
    tm *= 1000*1000*1000  # s to ns
    tm = np.diff(np.diff(tm, prepend=0), prepend=0)  # this adds 50 ms
    tm = tm.astype(np.uint32)
    tmss[key] = tm


In [ ]:
# zigzag each int to uint
for key, vl in vals.items():
    # vl = zigzag_encode(vl).astype(np.uint32)
    # and differentiate it? lets see the difference...
    #   looks like with random small data, it doesnt help too much
    #       makes sense
    #   could help in other contexts 
    vl = np.diff(vl, prepend=0)
    vl = np.diff(vl, prepend=0)
    vl = zigzag_encode(vl).astype(np.uint32)
    vals[key] = vl

In [ ]:
compressed_vals = {}
compressed_tmss = {}
for key, tm in tmss.items():
    # compress time
    compressed_tmss[key] = compress_array(tm)

for key, vl in vals.items():
    # compress vals
    compressed_vals[key] = compress_array(vl)


In [ ]:
testfil = '/home/fish2/mdf_compression/examples/high_random_values_doublediff.mdfc'

In [ ]:
# writer
with mdfc_writer(testfil) as writer:
    # each time/value --> this may not be too much higher than required compresion, 
    #   although definitely slower to decompress
    #   since conceptually only one time needs to be decompressed
    for key, ct in compressed_tmss.items():
        writer.append(ct, len(tmss[key]), f'time_{key}')
    for key, cv in compressed_vals.items():
        writer.append(cv, len(vals[key]), f'vals_{key}')
    
    writer.save()

    # fil = BytesIO(writer.fstream.getvalue())
    # fil_data = writer.fstream.getvalue()

In [ ]:
import sys
sys.path.append('/home/fish2/mdf_compression')
from utils import zigzag_encode, zigzag_decode, generate_uint32_buffer
from utils.compress import compress_array
from utils.decompress import decompress_array
from utils.serialize import mdfc_writer, mdfc_reader
import numpy as np

testfil = '/home/fish2/mdf_compression/examples/high_random_values.mdfc'

In [ ]:
%%timeit
all_datas = {}
with mdfc_reader(testfil) as reader:
    md = reader.metadata
    # buffer_array = None
    # it doesnt seem to change execution time much to allocate a buffer upfront, 
    #   vs allocate new one each time,
    #   so we can allocate once upfront and reuse for better global memory
    # TODO this should go into the decompressor
    max_shape = max(val[2] for val in md.values())
    buffer_array = generate_uint32_buffer(max_shape)
    
    for key in md.keys():
        ser = reader.load_series(key, buffer_array=buffer_array)  # adds ~55 ms, required
        if key.startswith('time_'):  # adds 75 ms :'(
            # TODO this can go into the decompressor somehow
            ser = ser.astype(np.int64)
            np.cumsum(ser, out=ser)
            np.cumsum(ser, out=ser)
        elif key.startswith('vals_'):  # does not add much time, ~5-10 ms
            ser = zigzag_decode(ser).astype(np.int32)
        all_datas[key] = ser


In [ ]:
sampl = all_datas['time_100ms_0']
n = len(sampl)
idxs = np.random.randint(0, n, 10000)
idxs.sort()

In [ ]:
%%timeit
sampl[idxs]
# yes, integer indexing is very fast, 
#   so only doing one double-prefix sum should save a lot of decompression time!

In [ ]:
(7.52*(len(all_datas)/2))/1000  # should cut down from 75 ms to ~5 ms --> big deal!

In [ ]:
# allocate one block with max decompressed size ever required


In [ ]:
vals  # 100 ms

In [ ]:
vals  # 1000 ms --> how does this have a higher compresed size?

In [ ]:
compress_array(vals)  # 1000 ms

In [ ]:
compress_array(vals)  # 100 ms

In [ ]:
# time axis becomes this many bytes of it
# so not that big of a deal to duplicate so many times
(495260 - 16)/1000/1000

In [ ]:
# values become these many
(17395916 - 495260)/1000/1000

In [ ]:
list(md.keys())[-1]

In [ ]:
4996/

In [ ]:
md['vals_100ms_5']

In [ ]:
md

In [ ]:
# we happen to know this needs more transforms
vals = zigzag_decode(vals).astype(np.int32)
times = np.cumsum(np.cumsum(times))

In [ ]:
vals

In [ ]:
times

In [ ]:
import numpy as np
arr = np.frombuffer(fil_data[16:16+532], dtype=np.uint32)
arr

In [ ]:
arr.shape

In [ ]:
decompress_array(arr, 6000)

In [ ]:
fil[-41:]

In [ ]:
# save the expected uncompressed size
size_of_vals = len(vals)
size_of_time = len(tmss)

In [ ]:
# codec --> TODO which one is "best" for each dtype, inteval, ... etc?
codec = getCodec('simdbinarypacking')
# to compress, 
# positional arguments:
#   input array, 
#   size of array
#   buffer for compressed data (of the same type? does it matter?)
#   size (element count) of buffer --> so i guess it doesnt matter?
buffer_vals = np.zeros(shape=vals.shape[0]+100, dtype=np.uint32)  # TODO: uint variant of dtype? dtype=vals.dtype)
# compress
compSize_vals = codec.encodeArray(vals, int(len(vals)), buffer_vals, int(len(buffer_vals)))
buffer_vals = buffer_vals[:compSize_vals]
buffer_vals.shape, vals.shape

In [ ]:
# take differential of time --> OK since we are always ascending in time, as per MDF standard
tmss = np.diff(tmss, prepend=0).astype(np.uint32)
# double differential of time would be more, 
#   any reason not to?
tmss = np.diff(tmss, prepend=0).astype(np.uint32)
tmss

In [ ]:
# codec = getCodec('simdbinarypacking')
buffer_tmss = np.zeros(shape=tmss.shape[0]+100, dtype=np.uint32)
compSize_tmss = codec.encodeArray(tmss, int(len(tmss)), buffer_tmss, int(len(buffer_tmss)))
buffer_tmss = buffer_tmss[:compSize_tmss]
buffer_tmss.shape, tmss.shape

In [ ]:
# "combined ratio" could be both... somehow...
f'simple example compression ratio: {((compSize_tmss + compSize_vals) / (len(tmss) + len(vals))):.3f}'

In [ ]:
# save these --> first, write the decompressed size, then the compressed array
with open('compressed_tmss', 'wb') as fil:
    fil.write(tmss_orig_len.to_bytes(8, signed=False))
    fil.write(buffer_tmss)
with open('compressed_vals', 'wb') as fil:
    fil.write(vals_orig_len.to_bytes(8, signed=False))
    fil.write(buffer_vals)

In [ ]:
import json

In [ ]:
len(json.dumps({'a': 1, 'b': 2}).encode('utf-8'))

In [ ]:
# test reading time
from pyfastpfor import *
import numpy as np
codec = getCodec('simdbinarypacking')

In [ ]:
%%timeit
# read it
with open('compressed_tmss', 'rb') as fil:
    decomp_size_tmss = int.from_bytes(fil.read(8))
    arr_tmss = fil.read()#sizeof_arr)
arr_tmss = np.frombuffer(arr_tmss, dtype=np.uint32)
with open('compressed_vals', 'rb') as fil:
    decomp_size_vals = int.from_bytes(fil.read(8))
    arr_vals = fil.read()#sizeof_arr)
arr_vals = np.frombuffer(arr_vals, dtype=np.uint32)

# decompress the values
# need:
#   compressed array, size of compressed array, 
#   buffer for decompression fill, size of decompression buffer
decomp_vals = np.zeros(shape=decomp_size_tmss, dtype=np.uint32)
codec.decodeArray(arr_vals, len(arr_vals), decomp_vals, len(decomp_vals))

# it has been zigzagged
# un zig
decomp_vals = (unsigned_right_shift(decomp_vals, 1)^ (- (decomp_vals & 1))).astype(np.int32)
# decomp_vals

# decompress the time
decomp_tmss = np.zeros(shape=decomp_size_tmss, dtype=np.uint32)
codec.decodeArray(arr_tmss, len(arr_tmss), decomp_tmss, len(decomp_tmss))

# it has been double differentiated
decomp_tmss = np.cumsum(np.cumsum(decomp_tmss))

In [ ]:
# wow so fast! 100x faster :) testing for a single channel
(
    11 /              # loading the single channel takes 11 ms
    (46.5 / 1000)     # 46.5 us to ms
)

In [ ]:
decomp_tmss

In [ ]:
decomp_vals

In [ ]:
(decomp_vals == vals_orig).all()

In [ ]:
(decomp_tmss == tmss_orig).all()

In [ ]:
# what about just zlibbing each transformed one...
#   large consistent integers seem to go better with zlib, 
#   maybe because of run length coding?
#   may be slower to decompress?
import zlib
(
zlib.compress(vals, level=9).__sizeof__() / vals.__sizeof__(),  # worse than fastpfor 
zlib.compress(tmss, level=9).__sizeof__() / tmss.__sizeof__(),  # better than fastpfor
)

In [ ]:
# what about zlib with no transformation...
import zlib
(
zlib.compress(vals_orig, level=9).__sizeof__() / vals.__sizeof__(),  # worse than fastpfor 
zlib.compress(tmss_orig, level=9).__sizeof__() / tmss.__sizeof__(),  # better than fastpfor
)

In [ ]:
# scratch below

In [ ]:
getCodecList()

In [ ]:
# original examples

In [ ]:
arrSize = 128 * 32
maxVal = 2048
# 1. Example without data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inp, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
assert(np.all(inpCompDecomp == inp))

In [ ]:
inp

In [ ]:
compSize

In [ ]:
arrSize

In [ ]:
inpComp[:compSize+3]

In [ ]:
arrSize = 128 * 32
maxVal = 1024 * 1024 * 1024 * 2

# 2. Example with slower data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

inp.sort()
inpCopy = np.array(inp, copy = True, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Carry out dafa differencing to convert a sorted sequence of large numbers
# into a sequence of small numbers (differences between adjacent numbers)
delta1(inpCopy, arrSize)


# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inpCopy, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
# Reverse differencing by computing the prefix sum
prefixSum1(inpCompDecomp, arrSize)

assert(np.all(inpCompDecomp == inp))

In [ ]:
arrSize = 128 * 32
maxVal = 1024 * 1024 * 1024 * 2

# 3. Example with faster but coarser data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

inp.sort()
inpCopy = np.array(inp, copy = True, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Carry out dafa differencing to convert a sorted sequence of large numbers
# into a sequence of small numbers (differences between numbers that are 4 indices apart)
delta4(inpCopy, arrSize)


# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inpCopy, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
# Reverse differencing by computing the prefix sum
prefixSum4(inpCompDecomp, arrSize)

assert(np.all(inpCompDecomp == inp))